In [2]:
import pandas as pd
from pathlib import Path

# ----------------------------------------
# 0) 경로 설정
# ----------------------------------------
PROJECT_ROOT = Path("/Users/gimgyumin/Desktop/Developer/commercial-area-analysis-ai")
RAW_DIR  = PROJECT_ROOT / "data" / "raw_data"
MAP_DIR  = PROJECT_ROOT / "data" / "category_maps"
OUT_DIR  = PROJECT_ROOT / "ai" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SALES_DIR = RAW_DIR / "sales_info"       # 매출_YYYY.csv
STORE_DIR = RAW_DIR / "store_info"       # YYYY-Q/상가_서울.csv
POP_PATH  = RAW_DIR / "population_info" / "유동인구.csv"

SALES_MAP_PATH = MAP_DIR / "sales_category_map.csv"
STORE_MAP_PATH = MAP_DIR / "store_category_map.csv"

OUT_PATH = OUT_DIR / "final_dataset.csv"


# ----------------------------------------
# 1) 한글 CSV 안전 로더
# ----------------------------------------
def read_csv_kor(path: Path, usecols=None, encodings=("utf-8-sig", "cp949", "euc-kr")):
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, usecols=usecols, low_memory=False)
        except Exception:
            continue
    raise RuntimeError(f"CSV 읽기 실패: {path}")


# ----------------------------------------
# 2) 카테고리 매핑 로드
# ----------------------------------------
sales_map_df = read_csv_kor(SALES_MAP_PATH)
store_map_df = read_csv_kor(STORE_MAP_PATH)

SALES_CATEGORY_MAP = dict(zip(sales_map_df["서비스_업종_코드_명"], sales_map_df["통합_카테고리"]))
STORE_CATEGORY_MAP = dict(zip(store_map_df["상권업종중분류명"].str.strip(), store_map_df["통합_카테고리"]))


# ----------------------------------------
# 3) 원본 로드
# ----------------------------------------

# 매출: 연도별 파일 전체 합치기
sales_df = pd.concat(
    [read_csv_kor(f) for f in sorted(SALES_DIR.glob("매출_*.csv"))],
    ignore_index=True
)

# 유동인구: 단일 파일
pop_df = read_csv_kor(POP_PATH)

# 상가: 분기별 폴더 순회 → 폴더명으로 기준_년분기_코드 생성
store_parts = []
for folder in sorted(STORE_DIR.iterdir()):
    if not folder.is_dir():
        continue
    store_file = folder / "상가_서울.csv"
    if not store_file.exists():
        print(f"파일 없음 (건너뜀): {folder.name}")
        continue
    quarter_code = int(folder.name.replace("-", ""))  # "2020-1" → 20201
    df = read_csv_kor(store_file)
    df["기준_년분기_코드"] = quarter_code
    store_parts.append(df)

store_df = pd.concat(store_parts, ignore_index=True)

print(f"매출 행 수: {len(sales_df):,}")
print(f"유동인구 행 수: {len(pop_df):,}")
print(f"상가 행 수: {len(store_df):,}")
print(f"상가 분기: {sorted(store_df['기준_년분기_코드'].unique())}")


# ----------------------------------------
# 4) 컬럼명 통일
# ----------------------------------------
sales_df = sales_df.rename(columns={"행정동_코드": "행정동코드", "행정동_코드_명": "행정동명"})
pop_df   = pop_df.rename(columns={"행정동_코드": "행정동코드", "행정동_코드_명": "행정동명"})

for df in (sales_df, pop_df, store_df):
    df["행정동코드"] = pd.to_numeric(df["행정동코드"], errors="coerce").astype("Int64")


# ----------------------------------------
# 5) 매출 집계 (분기 × 행정동 × 업종)
# ----------------------------------------
sales_df["통합_카테고리"] = sales_df["서비스_업종_코드_명"].map(SALES_CATEGORY_MAP)
sales_df = sales_df.dropna(subset=["통합_카테고리"]).copy()

sales_agg = (
    sales_df.groupby(["기준_년분기_코드", "행정동코드", "통합_카테고리"], dropna=False)
    .agg(
        당월매출합=("당월_매출_금액", "sum"),
        매출_20대합=("연령대_20_매출_금액", "sum"),
    )
    .reset_index()
)


# ----------------------------------------
# 6) 상가 집계 (분기 × 행정동 × 업종 → 점포수)
# ----------------------------------------
store_df["상권업종중분류명"] = store_df["상권업종중분류명"].str.strip()  # 공백 제거
store_df["통합_카테고리"] = store_df["상권업종중분류명"].map(STORE_CATEGORY_MAP)
store_df = store_df.dropna(subset=["통합_카테고리"]).copy()

store_agg = (
    store_df.groupby(["기준_년분기_코드", "행정동코드", "통합_카테고리"], dropna=False)
    .size()
    .reset_index(name="점포수")
)

dong_name = store_df[["행정동코드", "행정동명"]].drop_duplicates()


# ----------------------------------------
# 7) 유동인구 집계 (분기 × 행정동)
# ----------------------------------------
pop_agg = (
    pop_df.groupby(["기준_년분기_코드", "행정동코드"], dropna=False)
    .agg(
        총유동인구=("총_유동인구_수", "sum"),
        유동_20대=("연령대_20_유동인구_수", "sum"),
    )
    .reset_index()
)


# ----------------------------------------
# 8) 병합
# ----------------------------------------
merged = pd.merge(
    sales_agg,
    store_agg,
    on=["기준_년분기_코드", "행정동코드", "통합_카테고리"],
    how="left"
)

merged = pd.merge(
    merged,
    pop_agg,
    on=["기준_년분기_코드", "행정동코드"],
    how="left"
)

merged = pd.merge(
    merged,
    dong_name,
    on="행정동코드",
    how="left"
)


# ----------------------------------------
# 9) 행정동 전체 집계 파생 컬럼
# ----------------------------------------
dong_total = (
    merged.groupby(["기준_년분기_코드", "행정동코드"], dropna=False)
    .agg(
        행정동_전체매출=("당월매출합", "sum"),
        행정동_전체점포수=("점포수", "sum"),
    )
    .reset_index()
)

merged = pd.merge(merged, dong_total, on=["기준_년분기_코드", "행정동코드"], how="left")


# ----------------------------------------
# 10) 파생 feature 계산
# ----------------------------------------
merged["점포수"] = merged["점포수"].fillna(0)

merged["업종_점포당매출"]  = merged["당월매출합"] / merged["점포수"].replace(0, pd.NA)
merged["업종_매출점유율"]  = merged["당월매출합"] / merged["행정동_전체매출"].replace(0, pd.NA)
merged["업종_포화도"]      = merged["점포수"] / merged["행정동_전체점포수"].replace(0, pd.NA)
merged["경쟁강도"]         = merged["점포수"]
merged["매출_20대비율"]    = merged["매출_20대합"] / merged["당월매출합"].replace(0, pd.NA)
merged["유동_20대비율"]    = merged["유동_20대"] / merged["총유동인구"].replace(0, pd.NA)
merged["MZ_차이"]          = merged["매출_20대비율"] - merged["유동_20대비율"]
merged["유동대비매출"]     = merged["당월매출합"] / merged["총유동인구"].replace(0, pd.NA)
merged["점포대비유동"]     = merged["총유동인구"] / merged["점포수"].replace(0, pd.NA)

merged = merged.rename(columns={"통합_카테고리": "통합카테고리"})


# ----------------------------------------
# 11) 저장
# ----------------------------------------
col_order = [
    "기준_년분기_코드", "행정동코드", "통합카테고리",
    "당월매출합", "매출_20대합", "행정동명",
    "총유동인구", "유동_20대", "점포수",
    "행정동_전체매출", "행정동_전체점포수",
    "업종_점포당매출", "업종_매출점유율", "업종_포화도",
    "경쟁강도", "매출_20대비율", "유동_20대비율",
    "MZ_차이", "유동대비매출", "점포대비유동"
]
merged = merged[col_order].dropna(subset=["통합카테고리"])
merged.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"\n저장 완료: {OUT_PATH}")
print(f"행 수: {len(merged):,}")
print(f"컬럼 수: {merged.shape[1]}")
print(f"분기 범위: {sorted(merged['기준_년분기_코드'].unique())}")
print(merged.head(3))

매출 행 수: 458,616
유동인구 행 수: 11,475
상가 행 수: 12,180,927
상가 분기: [np.int64(20201), np.int64(20202), np.int64(20203), np.int64(20204), np.int64(20211), np.int64(20212), np.int64(20213), np.int64(20214), np.int64(20221), np.int64(20222), np.int64(20223), np.int64(20224), np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234), np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244), np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254)]

저장 완료: /Users/gimgyumin/Desktop/Developer/commercial-area-analysis-ai/ai/outputs/final_dataset.csv
행 수: 244,617
컬럼 수: 20
분기 범위: [np.int64(20191), np.int64(20192), np.int64(20193), np.int64(20194), np.int64(20201), np.int64(20202), np.int64(20203), np.int64(20204), np.int64(20211), np.int64(20212), np.int64(20213), np.int64(20214), np.int64(20221), np.int64(20222), np.int64(20223), np.int64(20224), np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234), np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20